<a href="https://colab.research.google.com/github/BosunL/SEA-VQA/blob/main/blip_vqa_base_SEA_CVQA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Visual Question Answering Example

This notebook demonstrates a basic Visual Question Answering (VQA) task using a pre-trained model from the Hugging Face Transformers library.

### Install Necessary Libraries

First, we need to install the `transformers` library to access pre-trained VQA models and `Pillow` for image processing.

In [ ]:
# Install transformers and other necessary libraries
!pip install -qqq transformers Pillow torch

### Import Libraries

Import the required modules from `transformers` and `PIL` (Pillow).

In [ ]:
from transformers import AutoProcessor, AutoModelForVisualQuestionAnswering
from PIL import Image
import requests
import io

### Load Pre-trained VQA Model and Processor

We'll use the `Salesforce/blip-vqa-base` model, which is a powerful VQA model.

In [ ]:
processor = AutoProcessor.from_pretrained("Salesforce/blip-vqa-base")
model = AutoModelForVisualQuestionAnswering.from_pretrained("Salesforce/blip-vqa-base")



### Model Evaluation

In a real-world scenario, evaluating a VQA model can be done in a few ways.

- **Qualitative Evaluation** (Manual Inspection): For a single example like the one demonstrated above, we can simply look at the image, the question, and the model's answer to see if it makes sense.

- **Quantitative Evaluation** (Using Datasets): This a more rigorous evaluation, you would typically use a VQA benchmark dataset. Such datasets contain thousands or millions of image-question pairs, each with multiple human-annotated answers. The common metrics are:

- **Accuracy**: The percentage of times the model's answer exactly matches one of the ground-truth answers.

- **VQA Score**: A more nuanced metric often used in VQA challenges. For each question, if the model's answer is present in the human answers, it gets a score that's min(human_answers_count / 3, 1). This accounts for cases where multiple correct answers exist.

**To perform quantitative evaluation, you would:**

Load a VQA dataset.
Iterate through the dataset, feeding each image-question pair to your model.
Collect the model's predictions.
Compare the predictions against the ground-truth answers using the chosen metric.


In [ ]:
# Lets try a real world VQA model Blip, and MCQA data

import requests
from PIL import Image
import io
import random
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (
    AutoProcessor,
    AutoModelForVisualQuestionAnswering,
    BlipProcessor,
    BlipForQuestionAnswering
)

In [ ]:
# ============================================================
# STEP 1: Load Model and Processor
# ============================================================
print("Loading model and processor...")

processor = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base")
model = BlipForQuestionAnswering.from_pretrained("Salesforce/blip-vqa-base")

print("Model loaded.\n")

##Using Google Drive for Direct Image Links

If you prefer to use Google Drive to host your images, follow these steps to get a direct download link:

1.  **Upload your image(s)** to Google Drive.
2.  **Right-click** on the image file and select **'Share'**.
3.  In the sharing dialog, change the access from 'Restricted' to **'Anyone with the link'**.
4.  **Copy the shareable link**. It will look something like this:
    `https://drive.google.com/file/d/FILE_ID/view?usp=sharing`
5.  **Convert the link to a direct download link** by extracting the `FILE_ID` and formatting it as:
    `https://drive.google.com/uc?export=download&id=FILE_ID`

    **Example:**
    If your shareable link is `https://drive.google.com/file/d/1aBcDeFGhIjKlMnOpQrStUvWxYz/view?usp=sharing`,
    your direct download link will be `https://drive.google.com/uc?export=download&id=1aBcDeFGhIjKlMnOpQrStUvWxYz`

6.  **Update your Google Sheet** with these direct download links in your `image_filename` column.

In [ ]:
import os
import pandas as pd
from datasets import Dataset, Features, Value, Image as ImageFeature

# 1. Configuration
GOOGLE_SHEET_URL = "https://docs.google.com/spreadsheets/d/e/2PACX-1vTs9PcGM6qO4MeGzL33BtqTDhFFt7Cq4YPXolFy-lpsdHfP-yBjL_EFvyC2Saoj_AJ9dm9VT9udAJgj/pub?output=csv"
IMAGE_DIR = "/content/images/"

def clean_image_path_or_url(filename):
    """Cleans up whitespace and handles absolute web URLs or local paths."""
    if not isinstance(filename, str):
        return filename
    filename = filename.strip()

    # If it is an internet URL, use it directly. If it is a filename, prepend the folder path.
    if filename.startswith(('http://', 'https://')):
        return filename
    return os.path.join(IMAGE_DIR, filename)

# 2. Extract Data from Google Sheet
print("Fetching Google Sheet...")
df = pd.read_csv(GOOGLE_SHEET_URL)
df.columns = df.columns.str.strip()  # Clean whitespace from headers

# Ensure an Index ID column is present
if 'ID' not in df.columns:
    df['ID'] = range(1, len(df) + 1)

# 3. Map Data Structures for Lazy Loading
if 'image_filename' in df.columns:
    # Set up final resource pointers
    df['image'] = df['image_filename'].apply(clean_image_path_or_url)
    df = df.drop(columns=['image_filename'])

    # Infer metadata schema types for the text columns
    features_dict = {
        col: Value(dtype='string') if dtype == 'object' else
             Value(dtype='int64') if 'int' in str(dtype) else
             Value(dtype='float64') if 'float' in str(dtype) else
             Value(dtype='bool') if 'bool' in str(dtype) else Value(dtype='string')
        for col, dtype in df.dtypes.items() if col != 'image'
    }
    features_dict['image'] = ImageFeature()  # Stream directly from path/URL string on-the-fly

    # Instantiate the stable dataset mapping
    print("Building Hugging Face lazy-loading structure...")
    custom_dataset_from_sheet = Dataset.from_pandas(df, features=Features(features_dict))

    samples = custom_dataset_from_sheet.select(range(len(custom_dataset_from_sheet)))
    print(f"✅ Loaded {len(samples)} samples successfully. System RAM footprint: ~0 MB.")
else:
    print("❌ Error: Column 'image_filename' was not found in your Google Sheet.")

After running this cell, `samples` will contain your data loaded from the Google Sheet, with the images processed accordingly. Remember to then uncomment and run cell `302bd8b0` to use this new `custom_dataset_from_sheet` for evaluation.

Now, the `samples` variable contains your custom data, and the subsequent evaluation steps will use this data.

In [ ]:
# To use your custom dataset, comment out the original `load_dataset` cell (JQVDEE5W8ZZR) and uncomment the following line:
dataset = custom_dataset_from_sheet
samples = dataset.select(range(len(custom_dataset_from_sheet)))

In [ ]:
# ============================================================
# STEP 2: Display Sample Images
# ============================================================
print("Displaying sample images...")

# Group samples by a unique identifier for the image.
# For this task, we'll use 'Category' as a grouping key, assuming all samples
# within the same 'Category' are intended to share the same visual context.
grouped_images = {}
for i, sample in enumerate(samples):
    category = sample["Category"] # Use Category for grouping
    if category not in grouped_images:
        grouped_images[category] = {
            "image": sample["image"], # Store the actual image object
            "question_indices": []    # Store original 1-based question numbers
        }
    grouped_images[category]["question_indices"].append(i + 1)

# Determine the number of unique images to plot
num_unique_images = len(grouped_images)
# Adjust figure size dynamically based on the number of unique images
fig, axes = plt.subplots(1, num_unique_images, figsize=(5 * num_unique_images, 4))

# Ensure 'axes' is always an iterable (list) for consistent indexing, even if only one subplot
if num_unique_images == 1:
    axes = [axes]

for idx, (category, group_info) in enumerate(grouped_images.items()):
    image_to_display = group_info["image"]
    question_numbers = group_info["question_indices"]

    # Format the question numbers for the title (e.g., "Q1", "Q1-4")
    q_label = f"Q{question_numbers[0]}"
    if len(question_numbers) > 1:
        q_label += f"-{question_numbers[-1]}"

    # Create the full title including the aggregated question label and category
    full_title = f"{q_label}\n{category}"

    axes[idx].imshow(image_to_display)
    axes[idx].set_title(full_title, fontsize=8)
    axes[idx].axis("off")

plt.suptitle("CVQA Images (English/Teso)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
def build_mcqa_prompt(question, choices):
    """
    Format question + choices into a prompt.
    Instructs the model to answer with only a letter A/B/C/D.
    """
    prompt = (
        "Look at the image carefully and answer the following "
        "multiple choice question by selecting only the letter "
        "A, B, C, or D.\n\n"
        f"Question: {question}\n"
    )
    for label, choice in zip(["A", "B", "C", "D"], choices):
        prompt += f"{label}. {choice}\n"
    prompt += "\nAnswer with only the letter (A, B, C, or D):"
    return prompt

def extract_predicted_label(predicted_text, choices):
    """
    Extract the predicted label (A/B/C/D) from model output.

    Strategy 1: look for A/B/C/D letter in model output.
    Strategy 2: if no letter found, check if model output
                matches any choice text directly.
    If no label is found, defaults to a random choice (A,B,C,D) to ensure a prediction is always made.
    """
    # Strategy 1 — look for letter
    for label in ["A", "B", "C", "D"]:
        if label in predicted_text.upper():
            return label

    # Strategy 2 — match answer text
    for label, choice in zip(["A", "B", "C", "D"], choices):
        if choice.lower() in predicted_text.lower():
            return label

    # Default to a random choice if no label is found
    import random
    return random.choice(["A", "B", "C", "D"])

###  Extend Evaluation to Teso Questions

Now, we'll extend the evaluation to also process questions in Teso. This requires specifying the column names in your Google Sheet that contain the Teso questions and their corresponding choices. The `evaluate_sea_mcqa` function will be updated to evaluate both English and Teso questions for each sample.

First, define the Teso-related column names here:

In [ ]:
# Define column names for Teso questions and answers
TESO_QUESTION_COLUMN = "native_question"
TESO_CORRECT_ANSWER_COLUMN = "correct_native"
TESO_WRONG_ANSWER_COLUMNS = ["wrong_native_o1", "wrong_native_o2", "wrong_native_o3"]

print(f"Teso Question Column: {TESO_QUESTION_COLUMN}")
print(f"Teso Correct Answer Column: {TESO_CORRECT_ANSWER_COLUMN}")
print(f"Teso Wrong Answer Columns: {TESO_WRONG_ANSWER_COLUMNS}")

print("\n")

# Define column names for Swahili questions and answers
SWA_QUESTION_COLUMN = "swa_question"
SWA_CORRECT_ANSWER_COLUMN = "correct_swa"
SWA_WRONG_ANSWER_COLUMNS = ["wrong_swa_o1", "wrong_swa_o2", "wrong_swa_o3"]

print(f"Swahili Question Column: {SWA_QUESTION_COLUMN}")
print(f"Swahili Correct Answer Column: {SWA_CORRECT_ANSWER_COLUMN}")
print(f"Swahili Wrong Answer Columns: {SWA_WRONG_ANSWER_COLUMNS}")

Now, the `evaluate_sea_mcqa` function will be modified to process both English and Teso questions for each sample. The results will include a `language` field to distinguish between them.

In [ ]:
# ============================================================
# STEP 3: Run Evaluation (Updated for Trilingual)
# ============================================================
def evaluate_sea_mcqa(samples, processor, model, teso_question_col, teso_correct_ans_col, teso_wrong_ans_cols, swa_question_col, swa_correct_ans_col, swa_wrong_ans_cols):
    """
    Evaluate model on SEA-MCQA for both English and Teso questions.
    Accuracy = % of samples where model selects the correct option.
    Random chance baseline = 25% (4 choices).
    """
    predictions = []

    def _evaluate_single_question_language(sample, question_col, correct_ans_col, wrong_ans_cols, language_tag):
        question    = sample[question_col]
        correct_ans = sample[correct_ans_col]
        wrong_1     = sample[wrong_ans_cols[0]]
        wrong_2     = sample[wrong_ans_cols[1]]
        wrong_3     = sample[wrong_ans_cols[2]]

        # Shuffle choices so correct answer isn't always in same position
        choices = [correct_ans, wrong_1, wrong_2, wrong_3]
        random.shuffle(choices)
        correct_label = ["A", "B", "C", "D"][choices.index(correct_ans)]

        # Build prompt
        prompt = build_mcqa_prompt(question, choices)

        predicted_text = "ERROR"
        predicted_label = None
        is_correct = False

        try:
            # Load image from dataset
            image = sample["image"].convert("RGB")

            # Run model
            inputs = processor(images=image, text=prompt, return_tensors="pt")
            outputs = model.generate(**inputs, max_new_tokens=10)
            predicted_text = processor.decode(
                outputs[0], skip_special_tokens=True
            ).strip()

            # Extract predicted label using both strategies
            predicted_label = extract_predicted_label(predicted_text, choices)

            # Check correctness
            is_correct = (predicted_label == correct_label)

        except Exception as e:
            print(f"Error processing sample ID {sample['ID']} ({language_tag} question): {e}")

        return {
            "sample_id":       sample["ID"],
            "category":        sample["Category"],
            "language":        language_tag,
            "question":        question,
            "choices":         {l: c for l, c in zip(["A","B","C","D"], choices)},
            "correct_answer":  correct_ans,
            "correct_label":   correct_label,
            "predicted_text":  predicted_text,
            "predicted_label": predicted_label,
            "is_correct":      is_correct
        }

    for i, sample in enumerate(samples):
        # Evaluate English question
        english_prediction = _evaluate_single_question_language(
            sample, "eng_question", "correct_en",
            ["wrong_en_o1", "wrong_en_o2", "wrong_en_o3"], "English"
        )
        predictions.append(english_prediction)

        # Evaluate Teso question
        teso_prediction = _evaluate_single_question_language(
            sample, teso_question_col, teso_correct_ans_col,
            teso_wrong_ans_cols, "Teso"
        )
        predictions.append(teso_prediction)


        # Evaluate Swahili question
        swa_prediction = _evaluate_single_question_language(
            sample, swa_question_col, swa_correct_ans_col,
            swa_wrong_ans_cols, "Swahili"
        )
        predictions.append(swa_prediction)


    overall_correct = sum(p['is_correct'] for p in predictions)
    overall_accuracy = (overall_correct / len(predictions)) * 100 if predictions else 0

    return overall_accuracy, predictions

# Call the updated evaluation function
accuracy, detailed_predictions = evaluate_sea_mcqa(
    samples, processor, model,
    TESO_QUESTION_COLUMN, TESO_CORRECT_ANSWER_COLUMN, TESO_WRONG_ANSWER_COLUMNS,
    SWA_QUESTION_COLUMN, SWA_CORRECT_ANSWER_COLUMN, SWA_WRONG_ANSWER_COLUMNS
)

print("Evaluation complete for English, Teso, and Swahili questions.")

### Print Results (Updated for Bilingual)

This section now displays the detailed evaluation results for both English and Teso questions.

In [ ]:
# ============================================================
# STEP 4: Print Results (Updated for Trilingual)
# ============================================================
print("\n" + "="*60)
print("          MCQA Evaluation Results (Englisg, Teso, and Swahili)")
print("="*60)

correct_english = sum(p['is_correct'] for p in detailed_predictions if p['language'] == 'English')
total_english = sum(1 for p in detailed_predictions if p['language'] == 'English')
accuracy_english = (correct_english / total_english) * 100 if total_english else 0

correct_teso = sum(p['is_correct'] for p in detailed_predictions if p['language'] == 'Teso')
total_teso = sum(1 for p in detailed_predictions if p['language'] == 'Teso')
accuracy_teso = (correct_teso / total_teso) * 100 if total_teso else 0


correct_swa = sum(p['is_correct'] for p in detailed_predictions if p['language'] == 'Swahili')
total_swa = sum(1 for p in detailed_predictions if p['language'] == 'Swahili')
accuracy_swa = (correct_swa / total_swa) * 100 if total_teso else 0


for i, pred in enumerate(detailed_predictions, 1):
    status = "✓" if pred["is_correct"] else "✗"
    print(f"\n[{status}] Q{pred['sample_id']} ({pred['language']}) [{pred['category']}]")
    print(f"  Question: {pred['question']}")
    for label, choice in pred["choices"].items():
        marker = " ← correct" if choice == pred["correct_answer"] else ""
        print(f"  {label}. {choice}{marker}")
    print(f"  Model output:    {pred['predicted_text']}")
    print(f"  Predicted label: {pred['predicted_label']}  "
          f"|  Correct label: {pred['correct_label']}")

print("\n" + "="*60)
print(f"Overall Accuracy:        {accuracy:.2f}% ({sum(p['is_correct'] for p in detailed_predictions)}/{len(detailed_predictions)}) ")
print(f"English Accuracy:        {accuracy_english:.2f}% ({correct_english}/{total_english})")
print(f"Teso Accuracy:           {accuracy_teso:.2f}% ({correct_teso}/{total_teso})")
print(f"Swahili Accuracy:        {accuracy_swa:.2f}% ({correct_swa}/{total_swa})")
print(f"Random chance baseline:  25.00% (4 choices)")
print("="*60)

###  Visualise Results (Updated for Trilingual)

This section now visualises the evaluation results, indicating correctness for both English and Teso questions.

In [ ]:
import matplotlib.pyplot as plt

# ============================================================
# STEP 5: Visualise Results (Updated for Trilingual: EN, TE, SW)
# ============================================================

num_samples = len(samples)

# We create a 4-row grid:
# Row 0: Image
# Row 1: English Prediction
# Row 2: Teso Prediction
# Row 3: Swahili Prediction
fig, axes = plt.subplots(4, num_samples, figsize=(4 * num_samples, 10), squeeze=False)

for i, sample in enumerate(samples):
    # Find predictions for this sample ID across all target languages
    sample_preds = [p for p in detailed_predictions if p['sample_id'] == sample['ID']]

    english_pred = next((p for p in sample_preds if p['language'] == 'English'), None)
    teso_pred    = next((p for p in sample_preds if p['language'] == 'Teso'), None)
    swahili_pred = next((p for p in sample_preds if p['language'] == 'Swahili'), None)

    # 1. Row 0: Display the input image
    axes[0, i].imshow(sample["image"])
    axes[0, i].set_title(f"Q{sample['ID']} [{sample.get('Category', 'N/A')}]", fontsize=10, fontweight='bold')
    axes[0, i].axis("off")

    # Helper function to plot text blocks cleanly
    def plot_lang_result(ax, lang_name, pred_data):
        ax.axis('off')
        if pred_data:
            color = "green" if pred_data["is_correct"] else "red"
            status = "✓" if pred_data["is_correct"] else "✗"
            text_str = (
                f"[{lang_name}] {status}\n"
                f"Pred: {pred_data['predicted_label']} | Correct: {pred_data['correct_label']}"
            )
            ax.text(
                0.5, 0.5, text_str,
                fontsize=8.5, color=color, ha='center', va='center',
                bbox=dict(facecolor='#f9f9f9', alpha=0.9, edgecolor=color, boxstyle='round,pad=0.5')
            )
        else:
            ax.text(
                0.5, 0.5, f"[{lang_name}]\nNo Data",
                fontsize=8.5, color='gray', ha='center', va='center',
                bbox=dict(facecolor='#eee', alpha=0.5, edgecolor='none', boxstyle='round,pad=0.5')
            )

    # 2. Row 1: English Results
    plot_lang_result(axes[1, i], "English", english_pred)

    # 3. Row 2: Teso Results
    plot_lang_result(axes[2, i], "Teso", teso_pred)

    # 4. Row 3: Swahili Results
    plot_lang_result(axes[3, i], "Swahili", swahili_pred)

# Set global title and layout boundaries
plt.suptitle(
    f"Trilingual Evaluation Results (English, Teso, Swahili)\n"
    f"Overall Accuracy: {accuracy:.1f}% | Random Baseline: 25.0%",
    fontsize=13, fontweight='bold'
)

plt.tight_layout(rect=[0, 0.02, 1, 0.94])
plt.show()

# ============================================================
# Printed Console Metrics Summary
# ============================================================
print("\n" + "="*55)
print("     MCQA Evaluation Results (English / Teso / Swahili)")
print("="*55)

# Breakdown by language in console output
for lang in ['English', 'Teso', 'Swahili']:
    lang_preds = [p for p in detailed_predictions if p.get('language') == lang]
    if lang_preds:
        lang_acc = (sum(p['is_correct'] for p in lang_preds) / len(lang_preds)) * 100
        print(f"{lang:<10} Accuracy:  {lang_acc:.2f}% ({sum(p['is_correct'] for p in lang_preds)}/{len(lang_preds)})")
    else:
        print(f"{lang:<10} Accuracy:  No Data")

print("-"*55)
print(f"Overall Accuracy:        {accuracy:.2f}%")
print(f"Total Correct:           {sum(p['is_correct'] for p in detailed_predictions)}/{len(detailed_predictions)}")
print(f"Random chance baseline:  25.00% (4 choices)")
print("="*55)